In [3]:
!pip install -q-spacy


Usage:   
  pip3 install [options] <requirement specifier> [package-index-options] ...
  pip3 install [options] -r <requirements file> [package-index-options] ...
  pip3 install [options] [-e] <vcs project url> ...
  pip3 install [options] [-e] <local project path> ...
  pip3 install [options] <archive url/path> ...

no such option: --


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [43]:

import spacy
import random
import json
import pandas as pd
import numpy as np
import string

from spacy.training.example import Example
from spacy.util import minibatch, compounding

from sklearn.model_selection import train_test_split

In [6]:
df = pd.read_csv("/content/drive/MyDrive/smart blind stick/sit_destination_ner_dataset.csv")

In [7]:
df.shape

(476, 6)

In [8]:
df.head()

,text,entity,start,end,label,canonical_location_reference
0,please guide me to birla auditorium,birla auditorium,19,35,DESTINATION,birla auditorium
1,i would like to visit birla auditorium,birla auditorium,22,38,DESTINATION,birla auditorium
2,where can i find birla auditorium,birla auditorium,17,33,DESTINATION,birla auditorium
3,what is the route to birla auditorium,birla auditorium,21,37,DESTINATION,birla auditorium
4,please tell me how to reach the birla auditorium,the birla auditorium,28,48,DESTINATION,birla auditorium


In [9]:
df.tail()

,text,entity,start,end,label,canonical_location_reference
471,please repeat the last sentence,NaN,-1,-1,O_ONLY,NaN
472,i am not asking for directions,NaN,-1,-1,O_ONLY,NaN
473,what are the college examination dates,NaN,-1,-1,O_ONLY,NaN
474,can you tell me a joke,NaN,-1,-1,O_ONLY,NaN
475,i need to send a message to my friend,NaN,-1,-1,O_ONLY,NaN


In [15]:
train_df , test_df = train_test_split(df , test_size = 0.2 , random_state = 42)

In [17]:
train_df.shape

(380, 6)

In [18]:
test_df.shape

(96, 6)

In [27]:
nlp = spacy.blank("en")
ner = nlp.add_pipe("ner")
ner.add_label("DESTINATION")

1

In [36]:
def prepare_data(data):

    examples = []

    for _, row in data.iterrows():

        doc = nlp.make_doc(row["text"])

        entities = []

        if row["label"] == "DESTINATION":

            start = int(row["start"])
            end = int(row["end"])

            # Extract the destination phrase
            destination = row["text"][start:end].strip()

            # Remove unnecessary articles
            for article in ["the ", "a ", "an "]:

                if destination.lower().startswith(article):
                    start += len(article)
                    destination = destination[len(article):]
                    break

            # Remove punctuation from the end
            while destination and destination[-1] in ".,?!":
                end -= 1
                destination = destination[:-1]

            entities = [
                (start, end, "DESTINATION")
            ]

        example = Example.from_dict(
            doc,
            {"entities": entities}
        )

        examples.append(example)

    return examples


train_examples = prepare_data(train_df)
test_examples = prepare_data(test_df)

print("Training examples:", len(train_examples))
print("Testing examples:", len(test_examples))

Training examples: 380
Testing examples: 96


In [37]:
optimizer = nlp.initialize(get_examples = lambda: train_examples)
for epoch in range(30) :
  random.shuffle(train_examples)
  losses = {}
  for example in train_examples :
    nlp.update(
        [example] ,
        sgd = optimizer ,
        drop = 0.2 ,
        losses = losses
    )
  print("Epoch :- " , epoch + 1 , "Losses :- " , losses)

Epoch :-  1 Losses :-  {'ner': np.float32(269.79013)}
Epoch :-  2 Losses :-  {'ner': np.float32(42.160248)}
Epoch :-  3 Losses :-  {'ner': np.float32(7.5960374)}
Epoch :-  4 Losses :-  {'ner': np.float32(7.2910028)}
Epoch :-  5 Losses :-  {'ner': np.float32(5.9153113)}
Epoch :-  6 Losses :-  {'ner': np.float32(30.360838)}
Epoch :-  7 Losses :-  {'ner': np.float32(6.355886)}
Epoch :-  8 Losses :-  {'ner': np.float32(22.030174)}
Epoch :-  9 Losses :-  {'ner': np.float32(11.648268)}
Epoch :-  10 Losses :-  {'ner': np.float32(1.5820427)}
Epoch :-  11 Losses :-  {'ner': np.float32(11.193972)}
Epoch :-  12 Losses :-  {'ner': np.float32(9.124628)}
Epoch :-  13 Losses :-  {'ner': np.float32(7.40036)}
Epoch :-  14 Losses :-  {'ner': np.float32(0.3664416)}
Epoch :-  15 Losses :-  {'ner': np.float32(16.868473)}
Epoch :-  16 Losses :-  {'ner': np.float32(26.207104)}
Epoch :-  17 Losses :-  {'ner': np.float32(4.674714)}
Epoch :-  18 Losses :-  {'ner': np.float32(0.00876897)}
Epoch :-  19 Losses :- 

In [38]:
scores = nlp.evaluate(test_examples)
print("Precision:", scores["ents_p"])
print("Recall:", scores["ents_r"])
print("F1 Score:", scores["ents_f"])

Precision: 0.9775280898876404
Recall: 1.0
F1 Score: 0.9886363636363636


In [41]:
nlp.to_disk("/content/drive/MyDrive/smart blind stick/models/sit_destination_ner_model")

In [44]:
def extract_destination(sentence):

    doc = nlp(sentence)

    for ent in doc.ents:

        if ent.label_ == "DESTINATION":

            destination = ent.text.strip()

            # Remove punctuation from beginning and end
            destination = destination.strip(string.punctuation)

            return destination

    return None

In [45]:
test_sentences = [
    "Take me to the ECE Block",
    "Can you guide me to the canteen?",
    "Take me to the Golden Jubilee Building",
    "Please guide me",
    "I want to go to the bank gate"
]

for sentence in test_sentences:

    destination = extract_destination(sentence)

    print("Input:", sentence)
    print("Destination:", destination)
    print("-" * 50)

Input: Take me to the ECE Block
Destination: ECE Block
--------------------------------------------------
Input: Can you guide me to the canteen?
Destination: canteen
--------------------------------------------------
Input: Take me to the Golden Jubilee Building
Destination: Golden Jubilee Building
--------------------------------------------------
Input: Please guide me
Destination: None
--------------------------------------------------
Input: I want to go to the bank gate
Destination: bank gate
--------------------------------------------------


In [47]:
model_path = "/content/drive/MyDrive/smart blind stick/models/sit_destination_ner_model"

nlp.to_disk(model_path)

print("NER model saved successfully!")

NER model saved successfully!


In [51]:
test_sentences = [
    "Take me to the ECE Block",
    "I want to visit Birla Auditorium",
    "Can you guide me to the canteen?",
    "Take me to the Golden Jubilee Building",
    "I need directions to the library",
    "Please guide me",
    "What is the weather today?",
    "I want to go to the bank gate"
]

for sentence in test_sentences:

    destination = extract_destination(sentence)

    print("Input:", sentence)
    print("Extracted destination:", destination)
    print("-" * 50)

Input: Take me to the ECE Block
Extracted destination: ECE Block
--------------------------------------------------
Input: I want to visit Birla Auditorium
Extracted destination: Birla Auditorium
--------------------------------------------------
Input: Can you guide me to the canteen?
Extracted destination: canteen
--------------------------------------------------
Input: Take me to the Golden Jubilee Building
Extracted destination: Golden Jubilee Building
--------------------------------------------------
Input: I need directions to the library
Extracted destination: library
--------------------------------------------------
Input: Please guide me
Extracted destination: None
--------------------------------------------------
Input: What is the weather today?
Extracted destination: None
--------------------------------------------------
Input: I want to go to the bank gate
Extracted destination: bank gate
--------------------------------------------------
